In [0]:
import json, urllib.request
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql import functions as F

latest = json.load(urllib.request.urlopen("https://stac.overturemaps.org/catalog.json"))["latest"]
base = f"s3://overturemaps-us-west-2/release/{latest}"
LANDING_DIR = "/Volumes/workspace/spb/landing"
MIN_CONFIDENCE = 0.4
H3_RES = 8

def flatten_places(df):
    """Pull fields out of the taxonomy and bbox structs."""
    return (
        df.withColumn("lon", F.col("bbox.xmin"))
          .withColumn("lat", F.col("bbox.ymin"))
          .drop("taxonomy", "bbox")
    )

def flatten_buildings(df):
    return (
          df.withColumn("lon", (F.col("bbox.xmin") + F.col("bbox.xmax")) / 2)
            .withColumn("lat", (F.col("bbox.ymin") + F.col("bbox.ymax")) / 2)
            .drop("bbox")
    )

def filter_to_bar(df):
    return df.where(
        (F.col("confidence") >= MIN_CONFIDENCE) &
        (F.col("basic_category") == "bar"))
    
def filter_places(df):
    return df.where(
        (F.col("confidence") >= MIN_CONFIDENCE))

def dedupe(df):
    return df.dropDuplicates(["name"])


def add_h3(df):
    """Built-in H3 index. Note the argument order: longitude, then latitude."""
    return df.withColumn("h3_cell", F.expr(f"h3_longlatash3string(lon, lat, {H3_RES})"))

In [0]:
bronze_places = spark.table("workspace.spb.bronze_places")
flat_places = flatten_places(bronze_places)
kept_places = filter_places(flat_places)
deduped_places = dedupe(kept_places)
silver_places = add_h3(deduped_places)

bars = filter_to_bar(flat_places)
dedupe_bar = dedupe(bars)
silver_bars = add_h3(dedupe_bar)

print("bronze", bronze_places.count())
print("after filter", kept_places.count())
print("after dedupe", deduped_places.count())


silver_places.write.format("delta").mode("overwrite").saveAsTable("workspace.spb.silver_places")
print("silver places", spark.table('workspace.spb.silver_places').count())

silver_bars.write.format("delta").mode("overwrite").saveAsTable("workspace.spb.silver_bars")
print("silver bars", spark.table('workspace.spb.silver_bars').count())

In [0]:
bronze_buildings = spark.table("workspace.spb.bronze_buildings")
flat_buldings = flatten_buildings(bronze_buildings)
deduped_buildings = dedupe(flat_buldings)
silver_buildings = add_h3(deduped_buildings)

silver_buildings.write.format("delta").mode("overwrite").saveAsTable("workspace.spb.silver_buildings")
print("silver buildings", spark.table("workspace.spb.silver_buildings").count())